# Introduction

In this notebook, we use the OpenClip model as transformer and trainform all the images into vectors, then feed the vectors to multiple ML algorithms.

## Convert Images into DataFrame

In [1]:
import torch
import os
import joblib

from PIL import Image
import open_clip
import numpy as np
import pandas as pd

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, roc_curve, auc

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


### Load OpenCLIP model

In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

### Load images and prepare data

In [4]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection-simple", 
                         ("train", "test", "val"))

In [5]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection-simple/val')]

In [5]:
len(train_images), len(test_images), len(valid_images)

(77449, 14961, 15274)

In [6]:
nazi_train_images = [image for image in train_images if "non-nazi" not in image]
nazi_test_images = [image for image in test_images if "non-nazi" not in image]
nazi_valid_images = [image for image in valid_images if "non-nazi" not in image]
len(nazi_train_images), len(nazi_test_images), len(nazi_valid_images)

(6118, 148, 301)

In [6]:
nazi_y_train = ["nazi-symbol"] * len(nazi_train_images)
nazi_y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
nazi_y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [7]:
y_train.count("nazi-symbol"), y_test.count("nazi-symbol"), y_valid.count("nazi-symbol")

(6118, 148, 301)

### Convert images to vectors and save as CSV

In [7]:
def load_image(image_path):
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [12]:
load_image(train_images[1])

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,-0.015738,0.093096,0.084752,0.112216,-0.000442,-0.065532,0.007091,-0.025539,-0.04525,0.040717,...,0.05709,-0.064796,-0.022393,0.011908,-0.00822,-0.018699,0.013882,0.019758,-0.008252,-0.018672


In [ ]:
with open("nazi_training_data_binary_reduced.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(nazi_train_images), 200):
    data = preprocess_images(nazi_train_images[i:i+200])
    with open("nazi_training_data_binary_reduced.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

In [ ]:
training_data = pd.read_csv("nazi_training_data_binary_reduced.csv")
training_data["label"] = "nazi-symbol"
training_data.to_csv("nazi_training_data_binary_reduced.csv", index=False)

In [ ]:
with open("nazi_validation_data_binary_reduced.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(nazi_valid_images), 200):
    data = preprocess_images(nazi_valid_images[i:i+200])
    with open("nazi_validation_data_binary_reduced.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

In [ ]:
validation_data = pd.read_csv("nazi_validation_data_binary_reduced.csv")
validation_data["label"] = "nazi-symbol"
validation_data.to_csv("nazi_validation_data_binary_reduced.csv", index=False)

In [ ]:
with open("nazi_test_data_binary_reduced.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(nazi_test_images), 200):
    data = preprocess_images(nazi_test_images[i:i+200])
    with open("nazi_test_data_binary_reduced.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

In [ ]:
test_data = pd.read_csv("nazi_test_data_binary_reduced.csv")
test_data["label"] = "nazi-symbol"
test_data.to_csv("nazi_test_data_binary_reduced.csv", index=False)

## Load data for training

In [8]:
training_data = pd.read_csv("training_data_binary_reduced.csv")
validation_data = pd.read_csv("validation_data_binary_reduced.csv")
test_data = pd.read_csv("test_data_binary_reduced.csv")

In [9]:
feature_columns = [str(i) for i in range(512)]
label_column = "label"

In [10]:
test_features = pd.concat([validation_data[feature_columns], test_data[feature_columns]])
test_labels = validation_data[label_column].tolist() + test_data[label_column].tolist()

In [11]:
y_true = [int(label == "nazi-symbol") for label in test_labels]

## Train and evaluate multiple classifiers

In [13]:
roc_auc_results = dict()

def gather_result(classifier):
    print("score on test: " + str(classifier.score(test_features, test_labels)))
    if getattr(classifier, "predict_proba", None):
        scores = pd.DataFrame(data=classifier.predict_proba(test_features), columns=classifier.classes_)
        outputs = [int(prob > 0.5) for prob in scores["nazi-symbol"]]
        fpr, tpr, threshold = roc_curve(y_true, scores["nazi-symbol"])
        roc_auc = auc(fpr, tpr)
        roc_auc_results[type(classifier).__name__] = dict(fpr=fpr, tpr=tpr, threshold=threshold, roc_auc=roc_auc)
    else:
        outputs = [int(label == "nazi-symbol") for label in classifier.predict(test_features)]
    print(classification_report(y_true, outputs, digits=3))
    print("accuracy score:", accuracy_score(y_true, outputs))
    print("roc auc score:", roc_auc_score(y_true, outputs))
    

In [12]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 6.05 s, sys: 6.46 s, total: 12.5 s
Wall time: 1.79 s


LogisticRegression(max_iter=5000)

In [14]:
gather_result(lr)

score on test: 0.9964279808169341
              precision    recall  f1-score   support

           0      0.998     0.999     0.998     29786
           1      0.917     0.835     0.874       449

    accuracy                          0.996     30235
   macro avg      0.957     0.917     0.936     30235
weighted avg      0.996     0.996     0.996     30235

accuracy score: 0.9964279808169341
roc auc score: 0.9170239168578473


In [15]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 544 ms, sys: 73.5 ms, total: 618 ms
Wall time: 762 ms


SGDClassifier()

In [16]:
gather_result(sgd)

score on test: 0.9969571688440549
              precision    recall  f1-score   support

           0      0.998     0.998     0.998     29786
           1      0.899     0.895     0.897       449

    accuracy                          0.997     30235
   macro avg      0.949     0.947     0.948     30235
weighted avg      0.997     0.997     0.997     30235

accuracy score: 0.9969571688440549
roc auc score: 0.9469060814956639


In [17]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 241 ms, sys: 75.7 ms, total: 316 ms
Wall time: 384 ms


KNeighborsClassifier(algorithm='brute', n_jobs=-1)

In [18]:
gather_result(knn)

score on test: 0.9981147676533818
              precision    recall  f1-score   support

           0      1.000     0.998     0.999     29786
           1      0.895     0.989     0.940       449

    accuracy                          0.998     30235
   macro avg      0.947     0.994     0.969     30235
weighted avg      0.998     0.998     0.998     30235

accuracy score: 0.9981147676533818
roc auc score: 0.9935591779639078


In [19]:
%%time

# import the library
from sklearn.svm import SVC

# instantiate & fit
svm=SVC(C=3)
svm.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 2min 14s, sys: 0 ns, total: 2min 14s
Wall time: 3min 1s


SVC(C=3)

In [20]:
gather_result(svm)

score on test: 0.9981478419050769
              precision    recall  f1-score   support

           0      0.998     1.000     0.999     29786
           1      0.973     0.900     0.935       449

    accuracy                          0.998     30235
   macro avg      0.986     0.950     0.967     30235
weighted avg      0.998     0.998     0.998     30235

accuracy score: 0.9981478419050769
roc auc score: 0.9497039909184403


In [21]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 18.3 s, sys: 72.3 ms, total: 18.4 s
Wall time: 25.2 s


DecisionTreeClassifier(max_depth=3, min_samples_split=10)

In [22]:
gather_result(clf)

score on test: 0.985645774764346
              precision    recall  f1-score   support

           0      0.994     0.991     0.993     29786
           1      0.514     0.635     0.568       449

    accuracy                          0.986     30235
   macro avg      0.754     0.813     0.780     30235
weighted avg      0.987     0.986     0.986     30235

accuracy score: 0.985645774764346
roc auc score: 0.8128396070140723


In [23]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 1min 13s, sys: 264 ms, total: 1min 13s
Wall time: 1min 37s


BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3,
                                                   min_samples_split=10),
                  max_samples=0.5)

In [24]:
gather_result(bg)

score on test: 0.9873656358524888
              precision    recall  f1-score   support

           0      0.995     0.993     0.994     29786
           1      0.566     0.639     0.600       449

    accuracy                          0.987     30235
   macro avg      0.780     0.816     0.797     30235
weighted avg      0.988     0.987     0.988     30235

accuracy score: 0.9873656358524888
roc auc score: 0.8159060989924116


In [25]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 20min 59s, sys: 4.57 s, total: 21min 3s
Wall time: 27min 58s


AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   learning_rate=0.5, n_estimators=100)

In [26]:
gather_result(adb)

score on test: 0.9945096742186208
              precision    recall  f1-score   support

           0      0.996     0.998     0.997     29786
           1      0.869     0.742     0.800       449

    accuracy                          0.995     30235
   macro avg      0.933     0.870     0.899     30235
weighted avg      0.994     0.995     0.994     30235

accuracy score: 0.9945096742186208
roc auc score: 0.8699847329659814


In [27]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 29min 27s, sys: 320 ms, total: 29min 27s
Wall time: 39min 33s


GradientBoostingClassifier()

In [28]:
gather_result(gbc)

score on test: 0.9936166694228543
              precision    recall  f1-score   support

           0      0.996     0.998     0.997     29786
           1      0.837     0.708     0.767       449

    accuracy                          0.994     30235
   macro avg      0.916     0.853     0.882     30235
weighted avg      0.993     0.994     0.993     30235

accuracy score: 0.9936166694228543
roc auc score: 0.8530795098577723


In [29]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 2min 31s, sys: 59 ms, total: 2min 31s
Wall time: 3min 23s


RandomForestClassifier(max_depth=3, n_estimators=300)

In [30]:
gather_result(rf)

score on test: 0.9881263436414751
              precision    recall  f1-score   support

           0      0.988     1.000     0.994     29786
           1      0.959     0.209     0.344       449

    accuracy                          0.988     30235
   macro avg      0.974     0.605     0.669     30235
weighted avg      0.988     0.988     0.984     30235

accuracy score: 0.9881263436414751
roc auc score: 0.6046099144947396


In [31]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC

evc=VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                                 ('rf', RandomForestClassifier(n_estimators=30,max_depth=3)),
                                 ('svm', SVC(max_iter=5000))])
evc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 2min 13s, sys: 8.09 s, total: 2min 21s
Wall time: 2min 51s


VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                             ('rf',
                              RandomForestClassifier(max_depth=3,
                                                     n_estimators=30)),
                             ('svm', SVC(max_iter=5000))])

In [32]:
gather_result(evc)

score on test: 0.9970563915991401
              precision    recall  f1-score   support

           0      0.997     1.000     0.999     29786
           1      0.964     0.833     0.894       449

    accuracy                          0.997     30235
   macro avg      0.981     0.916     0.946     30235
weighted avg      0.997     0.997     0.997     30235

accuracy score: 0.9970563915991401
roc auc score: 0.9162460593061986


In [33]:
joblib.dump(roc_auc_results, 'openclip-encoder-reduced-roc-auc-results.json')

['openclip-encoder-reduced-roc-auc-results.json']